In [12]:
import os
import cv2
import json
import numpy as np

CURR_DIR = os.getcwd()
WORKSPACE_DIR = os.path.dirname(CURR_DIR)
SAM2_DATASET_DIR = WORKSPACE_DIR + '/../sam2_labeled_data/ims_2024_day6_run2_vimba_rear_filtered/sam2_segmasks/'
FORMATTED_DATASET_DIR = WORKSPACE_DIR + '/../sam2_labeled_data/ims_2024_day6_run2_vimba_rear_filtered/labels/'
SAM2_IMAGES_DIR = WORKSPACE_DIR + '/../sam2_labeled_data/ims_2024_day6_run2_vimba_rear_filtered/images/'


In [44]:
def mask_to_polygon(mask):
    img_width = mask.shape[1]
    img_height = mask.shape[0]
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_TC89_L1)
    if len(contours) == 0:
        return []
    contours = [contour for contour in contours if len(contour) > 0]
    contours = [np.squeeze(contour, axis=1) for contour in contours]
    ret_contours = []
    for contour in contours:
        contour = np.array(contour, dtype=float)
        contour[:, 0] = contour[:, 0] / img_width
        contour[:, 1] = contour[:, 1] / img_height
        ret_contours.append(contour)
    # contours = np.array(contours, dtype=float)
    # # normalize for ultralytics format
    # contours[:, 0] = contours[:, 0] / img_width
    # contours[:, 1] = contours[:, 1] / img_height
    

    return ret_contours
mask_path = SAM2_DATASET_DIR + '/127.json'
with open(mask_path, "r") as file:
    data = json.load(file)
print(data.keys())
mask_array = np.array(data['1'], dtype=np.uint8)
print(mask_to_polygon(mask_array))

dict_keys(['1', '2', '3'])
[]


In [ ]:
# function that matches label name to corresponding img name
def format_sam2_label_name(label_name):
    # img name = 5 digit number, then .jpg
    # current label name = img number_1.json
    img_number = label_name.split('.')[0]
    # need to make sure img number is 5 digits
    img_number = img_number.zfill(6)
    return 'frame_' + img_number + '.txt'



def format_sam2_labels(dataset_dir, dest_dir):
    if not os.path.exists(dest_dir):
        os.makedirs(dest_dir)
    sorted_filenames = sorted(os.listdir(dataset_dir))
    for filename in sorted_filenames:
        if filename.endswith('.json'):
            with open(dataset_dir + '/' + filename, "r") as file:
                data = json.load(file)
            dest_label_name = format_sam2_label_name(filename)
            dest_filename = dest_dir + '/' + dest_label_name
            with open(dest_filename, "w") as file:
                string_to_write = ""
                for key in data.keys():
                    mask_array = np.array(data[key], dtype=np.uint8)
                    polygons = mask_to_polygon(mask_array)
                    for polygon in polygons:
                        if len(polygon) < 3:
                            continue
                        string_to_write += "2"
                        for point in polygon:
                            string_to_write += f" {point[0]} {point[1]}"
                        string_to_write += "\n"
                file.write(string_to_write)
            # print(f"Saved {dest_filename}")
                
format_sam2_labels(SAM2_DATASET_DIR, FORMATTED_DATASET_DIR)

0
0
0
0
0
0
0
0
0
0
0
0
1
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
1
2
0
1
0
1
0
1
2
0
0
1
0
0
0
0
1
0
1
0
1
2
0
1
0
1
2
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0


In [ ]:
# I was messing around and trying to make a video of the masks overlayed to see if the labels were right
def polygon_to_mask(label_path: os.PathLike, img_width: int, img_height: int) -> np.ndarray:
    with open(label_path, 'r') as f:
        # if there are any lines in the file, each represents a detection segmentation
        lines = f.readlines()
        mask = np.zeros((img_height, img_width), dtype=np.uint8)
        for line in lines:
            entries = line.split(' ')[1:]
            points = [float(entry) for entry in entries]

            # get rid of the first one because it's the detection class
            # the points are alternating x y x y, so we can just take the even and odd indices
            x_points = np.array([points[i]*img_width for i in range(0, len(points), 2)])
            y_points = np.array([points[i]*img_height for i in range(1, len(points), 2)])
            seg_vertices = np.column_stack((x_points, y_points)).astype(np.int32)
            cv2.fillPoly(mask, [seg_vertices], 1)
    return mask

def overlay_masks(img, masks, color=(0, 255, 0), alpha=0.5):
    overlay = img.copy()
    for mask in masks:
        overlay[mask == 1] = color
    return cv2.addWeighted(overlay, alpha, img, 1 - alpha, 0)

fps = 30
for file in sorted(os.listdir(SAM2_IMAGES_DIR))[35:85]:
    img_path = os.path.join(SAM2_IMAGES_DIR, file)
    
    img = cv2.imread(img_path)
    # if label doesnt exist, just use a blank array as mask
    # label_path = os.path.join(FORMATTED_DATASET_DIR, file.replace('.jpg', '.txt'))
    # true_mask = np.zeros((img.shape[0], img.shape[1]), dtype=np.uint8)
    # if os.path.exists(label_path):
    #     true_mask = polygon_to_mask(label_path, img.shape[1], img.shape[0])
    # overlay = overlay_masks(img, [true_mask])
    # cv2.imshow(f'file: {file}', overlay)
    # cv2.waitKey(int(1000/fps))
    # cv2.destroyAllWindows()




In [ ]:
# make a video out of these images to see the overlays
img_shape = img.shape
frame_width = img_shape[1]
frame_height = img_shape[0]
output_video = 'output.mp4'
fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Codec
video_writer = cv2.VideoWriter(output_video, fourcc, 10, (frame_width, frame_height))

for file in sorted(os.listdir(SAM2_IMAGES_DIR))[:300]:
    img_path = os.path.join(SAM2_IMAGES_DIR, file)
    
    img = cv2.imread(img_path)
    # if label doesnt exist, just use a blank array as mask
    label_path = os.path.join(FORMATTED_DATASET_DIR, file.replace('.PNG', '.txt'))
    print(label_path)
    
    true_mask = polygon_to_mask(label_path, img.shape[1], img.shape[0])
    overlay = overlay_masks(img, [true_mask])
    video_writer.write(overlay)

video_writer.release()
cv2.destroyAllWindows()

/home/tpark/Desktop/YOLOv8-Fine-Tune/../sam2_labeled_data/ims_2024_day6_run2_vimba_rear_filtered/labels/frame_000000.txt
/home/tpark/Desktop/YOLOv8-Fine-Tune/../sam2_labeled_data/ims_2024_day6_run2_vimba_rear_filtered/labels/frame_000001.txt
/home/tpark/Desktop/YOLOv8-Fine-Tune/../sam2_labeled_data/ims_2024_day6_run2_vimba_rear_filtered/labels/frame_000002.txt
/home/tpark/Desktop/YOLOv8-Fine-Tune/../sam2_labeled_data/ims_2024_day6_run2_vimba_rear_filtered/labels/frame_000003.txt
/home/tpark/Desktop/YOLOv8-Fine-Tune/../sam2_labeled_data/ims_2024_day6_run2_vimba_rear_filtered/labels/frame_000004.txt
/home/tpark/Desktop/YOLOv8-Fine-Tune/../sam2_labeled_data/ims_2024_day6_run2_vimba_rear_filtered/labels/frame_000005.txt
/home/tpark/Desktop/YOLOv8-Fine-Tune/../sam2_labeled_data/ims_2024_day6_run2_vimba_rear_filtered/labels/frame_000006.txt
/home/tpark/Desktop/YOLOv8-Fine-Tune/../sam2_labeled_data/ims_2024_day6_run2_vimba_rear_filtered/labels/frame_000007.txt
/home/tpark/Desktop/YOLOv8-Fine-